# mapping microwatershed with aquifer

Sanitised source snapshot. See the repository README and WORKFLOW.md before running. Outputs and execution metadata are removed. Historical experiments are not an unattended pipeline or reproduced scientific result.


In [ ]:
import os
from pathlib import Path

REPO_ROOT = next(parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / "prepare_wells.py").is_file())
WORKSPACE_ROOT = Path(os.environ.get("CGWB_WORKSPACE", str(REPO_ROOT / "data" / "cgwb"))).expanduser().resolve()
WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(WORKSPACE_ROOT)


In [ ]:
import pandas as pd

# Define the file path
file_path = f'{WORKSPACE_ROOT}/4. Mapping microwatershed with aquifer data/input/mws_with_representative_well.pickle'

# Load the DataFrame from the pickle file
mws_with_one_rep_well = pd.read_pickle(file_path)

# Display the loaded DataFrame
print(mws_with_one_rep_well.head())

In [ ]:
mws_with_one_rep_well

In [ ]:
import pandas as pd

# Define the file path
file_path = f'{WORKSPACE_ROOT}/4. Mapping microwatershed with aquifer data/input/aquifer_data_after_clubbing_categories.pickle'

# Load the DataFrame from the pickle file
loaded_aquifer_gdf = pd.read_pickle(file_path)

# Display the loaded DataFrame
print(loaded_aquifer_gdf.head())

In [ ]:
loaded_aquifer_gdf

In [ ]:
# Ensure both GeoDataFrames use the same CRS
crs_new_latest_filtered_gdf_with_one_well_per_mws = mws_with_one_rep_well.to_crs(loaded_aquifer_gdf.crs)

# Verify that 'loaded_gdf' contains well points
print("Loaded GeoDataFrame (well points) head:")
print(crs_new_latest_filtered_gdf_with_one_well_per_mws.head())

In [ ]:
import geopandas as gpd
import pandas as pd

In [ ]:
# Perform a spatial join to associate each well with the corresponding aquifer
final_gdf = gpd.sjoin(crs_new_latest_filtered_gdf_with_one_well_per_mws, loaded_aquifer_gdf, how='left', predicate='within')

# Display the results with relevant columns
print(final_gdf[['station_id', 'aquifer', 'aquifer_category']])

In [ ]:
for column in final_gdf.columns:
  print(column)

In [ ]:
# Merge to include the geometry of the microwatersheds
new_final_gdf = final_gdf.merge(loaded_aquifer_gdf[['id', 'geometry']], left_on='index_right', right_index=True, suffixes=('', '_aquifer'))

In [ ]:
new_final_gdf

In [ ]:
for column in new_final_gdf.columns:
  print(column)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Assuming 'cgwb_2001_onwards_renamed' is your dataframe
# The results dataframe already contains the necessary data
results = new_final_gdf[['station_id', 'consecutive_non_nan_pairs']]

# Calculate mean, median, and mode
mean_value = results['consecutive_non_nan_pairs'].mean()
median_value = results['consecutive_non_nan_pairs'].median()

print(f"Mean: {mean_value}")
print(f"Median: {median_value}")

# Calculate quartiles and percentiles
Q1 = results['consecutive_non_nan_pairs'].quantile(0.25)
Q3 = results['consecutive_non_nan_pairs'].quantile(0.75)
P10 = results['consecutive_non_nan_pairs'].quantile(0.10)
P20 = results['consecutive_non_nan_pairs'].quantile(0.20)

# Calculate the number of wells for each percentile and quartile
num_wells_P10 = results[results['consecutive_non_nan_pairs'] <= P10].shape[0]
num_wells_P20 = results[results['consecutive_non_nan_pairs'] <= P20].shape[0]
num_wells_Q1 = results[results['consecutive_non_nan_pairs'] <= Q1].shape[0]
num_wells_median = results[results['consecutive_non_nan_pairs'] <= median_value].shape[0]
num_wells_Q3 = results[results['consecutive_non_nan_pairs'] <= Q3].shape[0]

# Total number of wells
total_wells = results.shape[0]

# Plot histogram
plt.figure(figsize=(10, 6))
plt.hist(results['consecutive_non_nan_pairs'], bins=range(0, max(results['consecutive_non_nan_pairs']) + 5, 5), edgecolor='black')
plt.title('Histogram of number of consecutive Non zero pair of values present in the CGWB data (2001-2023)')
plt.xlabel('Number of consecutive Non zero pair of values')
plt.ylabel('Number of Wells')
plt.xticks(range(0, max(results['consecutive_non_nan_pairs']) + 5, 5))
plt.grid(axis='y')
plt.figtext(0.99, 0.01, f'Total number of wells: {total_wells}\n',
                         horizontalalignment='right', fontsize=10)
plt.show()

# Plot inverted box plot
plt.figure(figsize=(10, 6))
sns.boxplot(y=results['consecutive_non_nan_pairs'], color='lightyellow')
plt.title('Box Plot of number of consecutive Non zero pair of values present in the CGWB data (2001-2023)')
plt.ylabel('Number of consecutive Non zero pair of values')

# Annotate the quartiles and percentiles
plt.text(0.8, P10, f'10th Percentile: {P10:.2f} ({num_wells_P10} wells)', horizontalalignment='center', verticalalignment='center', color='green')
plt.text(0.8, P20, f'20th Percentile: {P20:.2f} ({num_wells_P20} wells)', horizontalalignment='center', verticalalignment='center', color='green')
plt.text(0.8, Q1, f'Q1: {Q1:.2f} ({num_wells_Q1} wells)', horizontalalignment='center', verticalalignment='center', color='blue')
plt.text(0.8, median_value, f'Median: {median_value:.2f} ({num_wells_median} wells)', horizontalalignment='center', verticalalignment='center', color='red')
plt.text(0.8, Q3, f'Q3: {Q3:.2f} ({num_wells_Q3} wells)', horizontalalignment='center', verticalalignment='center', color='blue')

# Remove x-axis label as it's not used in the plot
plt.xlabel('')
plt.figtext(0.99, 0.01, f'Total number of wells: {total_wells}\n'
                         f'The number of wells in brackets represents the number of wells whose data has the number of consecutive non-zero pairs of entries less than or equal to the value represented at the point.',
                         horizontalalignment='right', fontsize=10)
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming 'final_gdf' is your dataframe containing the necessary data
results = new_final_gdf[['station_id', 'consecutive_non_nan_pairs', 'aquifer_category']]

# Get the unique aquifer categories
unique_aquifer_categories = results['aquifer_category'].unique()

# Loop through each aquifer category and plot the histograms and box plots
for category in unique_aquifer_categories:
    category_data = results[results['aquifer_category'] == category]

    # Calculate mean, median, and quartiles
    mean_value = category_data['consecutive_non_nan_pairs'].mean()
    median_value = category_data['consecutive_non_nan_pairs'].median()
    Q1 = category_data['consecutive_non_nan_pairs'].quantile(0.25)
    Q3 = category_data['consecutive_non_nan_pairs'].quantile(0.75)
    P10 = category_data['consecutive_non_nan_pairs'].quantile(0.10)
    P20 = category_data['consecutive_non_nan_pairs'].quantile(0.20)

    # Calculate the number of wells for each percentile and quartile
    num_wells_P10 = category_data[category_data['consecutive_non_nan_pairs'] <= P10].shape[0]
    num_wells_P20 = category_data[category_data['consecutive_non_nan_pairs'] <= P20].shape[0]
    num_wells_Q1 = category_data[category_data['consecutive_non_nan_pairs'] <= Q1].shape[0]
    num_wells_median = category_data[category_data['consecutive_non_nan_pairs'] <= median_value].shape[0]
    num_wells_Q3 = category_data[category_data['consecutive_non_nan_pairs'] <= Q3].shape[0]

    # Total number of wells
    total_wells = category_data.shape[0]

    # Plot histogram
    plt.figure(figsize=(10, 6))
    plt.hist(category_data['consecutive_non_nan_pairs'], bins=range(0, max(category_data['consecutive_non_nan_pairs']) + 5, 5), edgecolor='black')
    plt.title(f'Histogram of number of consecutive Non zero pair of values\nin the CGWB data (2001-2023) for {category}')
    plt.xlabel('Number of consecutive Non zero pair of values')
    plt.ylabel('Number of Wells')
    plt.xticks(range(0, max(category_data['consecutive_non_nan_pairs']) + 5, 5))
    plt.grid(axis='y')
    plt.figtext(0.99, 0.01, f'Total number of wells: {total_wells}\n',
                             horizontalalignment='right', fontsize=10)
    plt.show()

    # Plot box plot
    plt.figure(figsize=(10, 6))
    sns.boxplot(y=category_data['consecutive_non_nan_pairs'], color='lightyellow')
    plt.title(f'Box Plot of number of consecutive Non zero pair of values\nin the CGWB data (2001-2023) for {category}')
    plt.ylabel('Number of consecutive Non zero pair of values')

    # Annotate the quartiles and percentiles
    plt.text(0.8, P10, f'10th Percentile: {P10:.2f} ({num_wells_P10} wells)', horizontalalignment='center', verticalalignment='center', color='green')
    plt.text(0.8, P20, f'20th Percentile: {P20:.2f} ({num_wells_P20} wells)', horizontalalignment='center', verticalalignment='center', color='green')
    plt.text(0.8, Q1, f'Q1: {Q1:.2f} ({num_wells_Q1} wells)', horizontalalignment='center', verticalalignment='center', color='blue')
    plt.text(0.8, median_value, f'Median: {median_value:.2f} ({num_wells_median} wells)', horizontalalignment='center', verticalalignment='center', color='red')
    plt.text(0.8, Q3, f'Q3: {Q3:.2f} ({num_wells_Q3} wells)', horizontalalignment='center', verticalalignment='center', color='blue')

    # Remove x-axis label as it's not used in the plot
    plt.xlabel('')
    plt.figtext(0.99, 0.01, f'Total number of wells: {total_wells}\n'
                             f'The number of wells in brackets represents the number of wells whose data has the number of consecutive non-zero pairs of entries less than or equal to the value represented at the point.',
                             horizontalalignment='right', fontsize=10)
    plt.show()

In [ ]:
new_final_gdf

In [ ]:
for column in new_final_gdf.columns:
  print(column)

In [ ]:
import geopandas as gpd

# Assuming new_latest_filtered_gdf_with_one_well_per_mws is your GeoDataFrame

# Define the file path
file_path = f'{WORKSPACE_ROOT}/4. Mapping microwatershed with aquifer data/output/mws_data_before_thresholding.pickle'

# Save the GeoDataFrame to a pickle file
new_final_gdf.to_pickle(file_path)

print(f"GeoDataFrame saved to {file_path}")